# Same-span KV versus text, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Uses all of `data/plan4_dev.jsonl` (30 families, 90 variants, 60 unique A sheets).
Do **not** load `plan4_fresh`.

A writes calculations once. The same complete spans are then sent two ways:

- `kv`: uncompressed full cache (mask/position control).
- `kv_result` / `text_result`: complete RESULT lines, shared positions on every head, then either gathered KV or those same tokens as text.
- `kv_probe` / `text_probe`: complete lines ranked by shared probe scores until at least as many tokens as the result spans; again KV versus the same text.

No extra sink. No headwise union for text. Frozen line-B. No budgets 32/64 or OBF.

Fresh four-way reference: compact text 78/90, fuller text 82/90, full KV 72/90, probe-256 40/90.
Development full KV was 82/90.

Download `/kaggle/working/results_stage_plan4_span`. Do not write into `results_stage_plan4_fresh` or earlier plan4 folders.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_span.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_span.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_span.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_span.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_dev", sum(1 for _ in (REPO / "data" / "plan4_dev.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_span")
print("Same selected spans: shared-position KV versus text. Uncompressed KV control. No 32/64 or OBF.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_span", "--split", "plan4_dev"])
print("plan4 span exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_span"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_span" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    print(path.read_text(encoding="utf-8")[:16000])